# PatchCore ViT-B/16 Block Depth Sweep (x224)

Sweeps ViT blocks [3, 6, 9, 11] with all other hyperparameters fixed at the main-experiment settings.
The goal is to justify the block-6 architectural choice used in the published result.

Artifacts written by this notebook:
- `experiments/anomaly_detection/patchcore/vit_b16/x224/block_depth_sweep/artifacts/block{N}/` - per-block scores and metrics
- `experiments/anomaly_detection/patchcore/vit_b16/x224/block_depth_sweep/artifacts/block_sweep_results.csv`
- `experiments/anomaly_detection/patchcore/vit_b16/x224/block_depth_sweep/artifacts/plots/block_sweep.png`


## Setup


In [1]:
import importlib, subprocess, sys
for pkg in ['timm', 'tqdm', 'scikit-learn']:
    if importlib.util.find_spec(pkg.replace('-', '_').split('scikit')[0] if 'scikit' in pkg else pkg) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('deps ready')


deps ready


## Imports


In [2]:
import gc, json, os, random, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import timm
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_fscore_support
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_CUDA = DEVICE.type == 'cuda'
if USE_CUDA:
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision('high')
print('Device:', DEVICE)


Device: cpu


## Configuration

All hyperparameters match the published main experiment (block 6, F1=0.595).
Only `VIT_FEATURE_BLOCK` varies across the sweep.


In [3]:
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / 'src' / 'wafer_defect').exists() and (candidate / 'configs').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('Could not locate repo root')
print('Repo root:', PROJECT_ROOT)

# -- Sweep parameters ----------------------------------------------------------
BLOCKS_TO_SWEEP = [3, 6, 9, 11]   # ViT-B/16 has 12 blocks (0-indexed)

# -- Fixed hyperparameters (identical to main experiment) ----------------------
DATA_PATH            = str(PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl')
IMAGE_SIZE           = 224
PATCH_EMBED_DIM      = 128          # single-block projection dim
MEMORY_BANK_MAX      = 400_000
TOPK_PATCH_RATIO     = 0.10
PATCHCORE_NN_K       = 3
SCORE_CHUNK          = 512
THRESHOLD_QUANTILE   = 0.95         # val-normal-only threshold

TRAIN_NORMAL_N = 40_000
VAL_NORMAL_N   =  5_000
TEST_NORMAL_N  =  5_000
TEST_DEFECT_N  =    250

BATCH_SIZE  = 128
NUM_WORKERS = 0

ARTIFACT_BASE = PROJECT_ROOT / 'experiments/anomaly_detection/patchcore/vit_b16/x224/block_depth_sweep/artifacts'
PLOTS_DIR = ARTIFACT_BASE / 'plots'
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# If True, re-runs even when a saved scores.npz exists for that block.
FORCE_RERUN = False

print(f'Sweeping blocks: {BLOCKS_TO_SWEEP}')
print(f'Artifacts root: {ARTIFACT_BASE}')


Repo root: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project
Sweeping blocks: [3, 6, 9, 11]
Artifacts root: C:\Users\genso\Documents\College_Projects\DeepLearning 2610\Project\experiments\anomaly_detection\patchcore\vit_b16\x224\block_depth_sweep\artifacts


## Load Dataset

Data is loaded once and shared across all block experiments.


In [4]:
SRC_ROOT = PROJECT_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from wafer_defect.data.legacy_pickle import read_legacy_pickle

df_raw = read_legacy_pickle(DATA_PATH)

def parse_failure_label(v):
    if v is None: return 'unknown'
    if isinstance(v, float) and np.isnan(v): return 'unknown'
    if isinstance(v, (list, tuple, np.ndarray)):
        a = np.array(v).reshape(-1)
        return 'unknown' if len(a) == 0 else str(a[0])
    return str(v)

df = df_raw.copy()
df['failure_label'] = df['failureType'].apply(parse_failure_label).str.strip()
invalid = {'0', 'unknown', 'nan', 'None', '[]'}
df = df[~df['failure_label'].isin(invalid)].copy()
df['is_anomaly'] = (df['failure_label'].str.lower() != 'none').astype(int)

normal_df = df[df['is_anomaly'] == 0].copy()
defect_df = df[df['is_anomaly'] == 1].copy()
print(f'Labeled: {len(df):,}  Normal: {len(normal_df):,}  Defect: {len(defect_df):,}')

rng = np.random.default_rng(SEED)
ns = normal_df.iloc[rng.permutation(len(normal_df))].reset_index(drop=True)
ds = defect_df.iloc[rng.permutation(len(defect_df))].reset_index(drop=True)

a = TRAIN_NORMAL_N
b = a + VAL_NORMAL_N
c = b + TEST_NORMAL_N

train_normal_df = ns.iloc[0:a].copy()
val_normal_df   = ns.iloc[a:b].copy()
test_normal_df  = ns.iloc[b:c].copy()
test_defect_df  = ds.iloc[0:TEST_DEFECT_N].copy()

del df_raw, df, normal_df, defect_df, ns, ds
gc.collect()

print(f'Train normal : {len(train_normal_df):>7,}')
print(f'Val normal   : {len(val_normal_df):>7,}')
print(f'Test normal  : {len(test_normal_df):>7,}')
print(f'Test defect  : {len(test_defect_df):>7,}')


Labeled: 172,950  Normal: 147,431  Defect: 25,519
Train normal :  40,000
Val normal   :   5,000
Test normal  :   5,000
Test defect  :     250


## Dataset Class and Helpers


In [5]:
class WaferDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, size: int = 224):
        self.maps   = frame['waferMap'].values
        self.labels = frame['is_anomaly'].values.astype(np.int64)
        self.size   = size

    def __len__(self):
        return len(self.maps)

    def __getitem__(self, idx):
        arr = np.clip(np.array(self.maps[idx], dtype=np.int64), 0, 2)
        x   = torch.tensor(arr, dtype=torch.long)
        x   = F.one_hot(x, num_classes=3).permute(2, 0, 1).float()
        x   = F.interpolate(x.unsqueeze(0), size=(self.size, self.size), mode='nearest').squeeze(0)
        return x, int(self.labels[idx])


loader_kw = dict(batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
                 pin_memory=USE_CUDA, persistent_workers=(NUM_WORKERS > 0))

train_loader     = DataLoader(WaferDataset(train_normal_df, IMAGE_SIZE), **loader_kw)
val_loader       = DataLoader(WaferDataset(val_normal_df,   IMAGE_SIZE), **loader_kw)
test_normal_loader = DataLoader(WaferDataset(test_normal_df, IMAGE_SIZE), **loader_kw)
test_defect_loader = DataLoader(WaferDataset(test_defect_df, IMAGE_SIZE), **loader_kw)

xb, _ = next(iter(train_loader))
print(f'Batch shape: {tuple(xb.shape)}')


def build_extractor(block_idx: int, proj_dim: int = PATCH_EMBED_DIM):
    """Single-block ViT-B/16 feature extractor."""
    class ViTExtractor(nn.Module):
        def __init__(self):
            super().__init__()
            self.vit = timm.create_model(
                'vit_base_patch16_224.augreg_in21k_ft_in1k',
                pretrained=True, num_classes=0,
            )
            self._feat = {}
            self.vit.blocks[block_idx].register_forward_hook(
                lambda m, i, o: self._feat.__setitem__('out', o)
            )
            self.proj = nn.Linear(768, proj_dim, bias=False)

        def forward(self, x):
            self._feat = {}
            self.vit(x)
            toks = self._feat['out'][:, 1:, :]   # drop CLS -> [B, 196, 768]
            return self.proj(toks)               # [B, 196, proj_dim]

    model = ViTExtractor().to(DEVICE).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model


def extract_embeddings(extractor, xb):
    with torch.inference_mode():
        with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
            emb = extractor(xb)
        emb = emb.float().reshape(-1, emb.shape[-1])
        emb = F.normalize(emb, p=2, dim=1)
    return emb


def build_memory_bank(extractor, loader):
    sampled, total_seen, ratio = [], 0, None
    for step, (xb, _) in enumerate(tqdm(loader, desc='Bank build', unit='batch')):
        xb  = xb.to(DEVICE)
        emb = extract_embeddings(extractor, xb)
        total_seen += len(emb)
        if ratio is None:
            est_total = (len(emb) // len(xb)) * len(train_normal_df)
            ratio = min(1.0, MEMORY_BANK_MAX / est_total)
        if ratio < 1.0:
            k   = max(1, int(round(len(emb) * ratio)))
            idx = torch.randperm(len(emb), device=DEVICE)[:k]
            emb = emb[idx]
        sampled.append(emb)
    bank = torch.cat(sampled, dim=0)
    if len(bank) > MEMORY_BANK_MAX:
        bank = bank[torch.randperm(len(bank), device=DEVICE)[:MEMORY_BANK_MAX]]
    bank = F.normalize(bank, p=2, dim=1).contiguous()
    return bank


def min_dist_to_bank(patches, bank_t, chunk=512, k=3):
    out = []
    for i in range(0, len(patches), chunk):
        p   = patches[i:i+chunk]
        sim = p @ bank_t
        kk  = min(k, sim.shape[1])
        best = sim.topk(kk, dim=1).values
        dist = torch.sqrt(torch.clamp(2.0 - 2.0 * best, min=0.0)).mean(dim=1)
        out.append(dist)
    return torch.cat(out)


def score_loader_fn(extractor, loader, bank_t, topk_ratio=TOPK_PATCH_RATIO,
                    nn_k=PATCHCORE_NN_K, desc='Scoring'):
    scores = []
    with torch.inference_mode():
        for xb, _ in tqdm(loader, total=len(loader), desc=desc, unit='batch'):
            xb = xb.to(DEVICE)
            with torch.autocast('cuda', torch.float16, enabled=USE_CUDA):
                emb = extractor(xb)
            emb = emb.float().reshape(-1, emb.shape[-1])
            emb = F.normalize(emb, p=2, dim=1)
            ps  = min_dist_to_bank(emb, bank_t, SCORE_CHUNK, nn_k)
            b   = len(xb)
            ps  = ps.reshape(b, -1)
            topk = max(1, min(int(round(ps.shape[1] * topk_ratio)), ps.shape[1]))
            s   = ps.topk(topk, dim=1).values.mean(dim=1)
            scores.append(s.cpu())
    return torch.cat(scores).numpy()


def compute_metrics(val_normal_scores, test_normal_scores, test_defect_scores,
                    threshold_q=THRESHOLD_QUANTILE):
    threshold = float(np.quantile(val_normal_scores, threshold_q))
    y_true = np.concatenate([np.zeros(len(test_normal_scores)), np.ones(len(test_defect_scores))])
    scores = np.concatenate([test_normal_scores, test_defect_scores])
    y_pred = (scores > threshold).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average='binary', zero_division=0)
    auroc = float(roc_auc_score(y_true, scores))
    auprc = float(average_precision_score(y_true, scores))
    return dict(threshold=threshold, precision=float(p), recall=float(r),
                f1=float(f1), auroc=auroc, auprc=auprc)

print('Helpers defined.')


Batch shape: (128, 3, 224, 224)
Helpers defined.


## Block Sweep

Iterates over `BLOCKS_TO_SWEEP`. Each block:
1. Loads saved `scores.npz` if it exists (and `FORCE_RERUN=False`).
2. Otherwise builds a fresh memory bank and scores all splits.
3. Saves `scores.npz` and `metrics.json` to `artifacts/block{N}/`.


In [ ]:
sweep_rows = []

for block_idx in BLOCKS_TO_SWEEP:
    print(f'\n{'='*60}')
    print(f'Block {block_idx}')
    print(f'{'='*60}')

    block_dir = ARTIFACT_BASE / f'block{block_idx}'
    block_dir.mkdir(parents=True, exist_ok=True)
    scores_path  = block_dir / 'scores.npz'
    metrics_path = block_dir / 'metrics.json'

    if scores_path.exists() and not FORCE_RERUN:
        print(f'  Loading saved scores from {scores_path}')
        d = np.load(scores_path)
        val_n_z   = d['val_normal_z']
        test_n_z  = d['test_normal_z']
        test_d_z  = d['test_defect_z']
    else:
        print(f'  Building extractor for block {block_idx}..')
        extractor = build_extractor(block_idx)

        print('  Building memory bank..')
        bank = build_memory_bank(extractor, train_loader)
        bank_t = bank.t().contiguous()
        print(f'  Bank: {len(bank):,} x {bank.shape[1]}-d')

        train_scores = score_loader_fn(extractor, train_loader, bank_t, desc=f'Score train  [blk{block_idx}]')
        val_scores   = score_loader_fn(extractor, val_loader,   bank_t, desc=f'Score val    [blk{block_idx}]')
        test_n_scores = score_loader_fn(extractor, test_normal_loader, bank_t, desc=f'Score test-n [blk{block_idx}]')
        test_d_scores = score_loader_fn(extractor, test_defect_loader, bank_t, desc=f'Score test-d [blk{block_idx}]')

        mu  = float(np.mean(train_scores))
        std = float(np.std(train_scores) + 1e-8)
        val_n_z  = (val_scores    - mu) / std
        test_n_z = (test_n_scores - mu) / std
        test_d_z = (test_d_scores - mu) / std

        np.savez_compressed(
            scores_path,
            val_normal_z=val_n_z, test_normal_z=test_n_z, test_defect_z=test_d_z,
            train_mu=np.array(mu), train_std=np.array(std),
        )
        print(f'  Scores saved to {scores_path}')

        del extractor, bank, bank_t
        gc.collect()
        if USE_CUDA:
            torch.cuda.empty_cache()

    metrics = compute_metrics(val_n_z, test_n_z, test_d_z)
    metrics['block_idx'] = block_idx
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')

    print(f'  F1={metrics["f1"]:.4f}  AUROC={metrics["auroc"]:.4f}  AUPRC={metrics["auprc"]:.4f}')
    sweep_rows.append(metrics)


sweep_df = pd.DataFrame(sweep_rows).sort_values('block_idx').reset_index(drop=True)
sweep_csv = ARTIFACT_BASE / 'block_sweep_results.csv'
sweep_df.to_csv(sweep_csv, index=False)
print(f'\nSweep results saved to {sweep_csv}')
sweep_df[['block_idx', 'f1', 'auroc', 'auprc', 'threshold', 'precision', 'recall']]



Block 3
  Building extractor for block 3..


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

  Building memory bank..


Bank build:   0%|          | 0/313 [00:00<?, ?batch/s]

## Plot: Block Sweep Comparison


In [ ]:
from IPython.display import display

display(sweep_df[['block_idx', 'f1', 'auroc', 'auprc', 'precision', 'recall']].round(4))

x     = np.arange(len(sweep_df))
width = 0.25
labels = [f'Block {int(b)}' for b in sweep_df['block_idx']]

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(x - width, sweep_df['f1'],    width, label='F1',    color='#3d405b')
ax.bar(x,         sweep_df['auroc'], width, label='AUROC', color='#81b29a')
ax.bar(x + width, sweep_df['auprc'], width, label='AUPRC', color='#f2cc8f')

for i, row in sweep_df.iterrows():
    ax.text(x[i] - width, row['f1']    + 0.005, f'{row["f1"]:.3f}',    ha='center', va='bottom', fontsize=8)
    ax.text(x[i],         row['auroc'] + 0.005, f'{row["auroc"]:.3f}', ha='center', va='bottom', fontsize=8)
    ax.text(x[i] + width, row['auprc'] + 0.005, f'{row["auprc"]:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0.0, 1.05)
ax.set_ylabel('Score')
ax.set_title('ViT-B/16 PatchCore - Block Depth Sweep (x224, 95th-pct val-normal threshold)')
ax.legend()
plt.tight_layout()
fig.savefig(PLOTS_DIR / 'block_sweep.png', dpi=200, bbox_inches='tight')
plt.show()
plt.close(fig)
print(f'Saved: {PLOTS_DIR / "block_sweep.png"}')


## Save Outputs


In [ ]:
# Standardized local artifact export

import shutil

checkpoints_dir = ARTIFACT_BASE / 'checkpoints'
results_dir = ARTIFACT_BASE / 'results'
plots_dir = ARTIFACT_BASE / 'plots'
for _path in [checkpoints_dir, results_dir, plots_dir]:
    _path.mkdir(parents=True, exist_ok=True)

best_row = sweep_df.sort_values(['f1', 'auroc', 'auprc'], ascending=False).iloc[0]
best_block = int(best_row['block_idx'])
best_scores_path = ARTIFACT_BASE / f'block{best_block}' / 'scores.npz'
best_metrics_path = ARTIFACT_BASE / f'block{best_block}' / 'metrics.json'

with np.load(best_scores_path) as d:
    test_normal_scores_z = d['test_normal_z']
    test_defect_scores_z = d['test_defect_z']
    val_normal_scores_z = d['val_normal_z']
    mu = float(d['train_mu'])
    std = float(d['train_std'])

threshold_z = float(best_row['threshold'])
threshold_raw = mu + threshold_z * std
y_true = np.concatenate([np.zeros(len(test_normal_scores_z), dtype=int), np.ones(len(test_defect_scores_z), dtype=int)])
y_pred = (np.concatenate([test_normal_scores_z, test_defect_scores_z]) > threshold_z).astype(int)
test_scores_df = pd.concat([
    test_normal_df[['failure_label', 'is_anomaly']].assign(split='test_normal'),
    test_defect_df[['failure_label', 'is_anomaly']].assign(split='test_defect'),
], ignore_index=True)
test_scores_df['score_z'] = np.concatenate([test_normal_scores_z, test_defect_scores_z])
test_scores_df['predicted_anomaly'] = y_pred
test_scores_df.to_csv(results_dir / 'test_scores.csv', index=False)

defect_recall_df = (
    test_scores_df.loc[test_scores_df['is_anomaly'] == 1]
  .groupby('failure_label')
  .agg(count=('predicted_anomaly', 'count'), detected=('predicted_anomaly', 'sum'), recall=('predicted_anomaly', 'mean'), mean_score=('score_z', 'mean'))
  .reset_index()
)
defect_recall_df.to_csv(results_dir / 'test_defect_recall.csv', index=False)

score_columns = []
for _block in sweep_df['block_idx'].astype(int).tolist():
    with np.load(ARTIFACT_BASE / f'block{_block}' / 'scores.npz') as _d:
        score_columns.append(np.concatenate([_d['test_normal_z'], _d['test_defect_z']]))
umap_matrix = np.column_stack(score_columns)
umap_df = test_scores_df[['failure_label', 'is_anomaly', 'split', 'score_z', 'predicted_anomaly']].copy()
umap_df.insert(0, 'point_index', np.arange(len(umap_df)))

try:
    import umap.umap_ as umap
except Exception:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'umap-learn', '-q'])
    import umap.umap_ as umap

coords = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, metric='euclidean', random_state=42, transform_seed=42, low_memory=True).fit_transform(umap_matrix)
umap_df['umap_1'] = coords[:, 0]
umap_df['umap_2'] = coords[:, 1]
umap_df.to_csv(results_dir / 'umap_test_embeddings.csv', index=False)

fig, ax = plt.subplots(figsize=(8, 6))
normal_mask = umap_df['is_anomaly'].to_numpy() == 0
ax.scatter(coords[normal_mask, 0], coords[normal_mask, 1], s=8, alpha=0.45, c='steelblue', label='Normal', linewidths=0)
ax.scatter(coords[~normal_mask, 0], coords[~normal_mask, 1], s=8, alpha=0.60, c='tomato', label='Defect', linewidths=0)
ax.set_title('UMAP of Test Scores Across Sweep Blocks')
ax.set_xlabel('UMAP-1')
ax.set_ylabel('UMAP-2')
ax.legend()
fig.tight_layout()
fig.savefig(plots_dir / 'umap_test_embeddings.png', dpi=160, bbox_inches='tight')
plt.show()
plt.close(fig)

artifact = {
    'selected_block': best_block,
    'threshold_z': threshold_z,
    'threshold_raw': threshold_raw,
    'source_scores_path': str(best_scores_path),
    'source_metrics_path': str(best_metrics_path),
    'block_results_path': str(ARTIFACT_BASE / 'block_sweep_results.csv'),
}
torch.save(artifact, checkpoints_dir / 'model.pt')
(results_dir / 'evaluation_metrics.json').write_text(json.dumps(artifact | json.loads(best_metrics_path.read_text(encoding='utf-8')), indent=2), encoding='utf-8')
